# 1. Setup

Combine 3Cv1 joint, crop and disease paths. Gather primitive logits using each candidate’s IDs and normalize each path over candidates. Inference-only experiment inspired by [Troika](https://arxiv.org/abs/2303.15230). Set the run folder(s) to the local checkpoint directories containing selection.json and the .pt weights. For the ensemble, the publisher page was unavailable when preparing this notebook; no paper metrics are assumed.

In [1]:
import json, math, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from timm.layers import set_reentrant_ckpt
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split, get_final_test_loaders
from common.evaluate import evaluate, val_accuracy, concept_accuracy
MODEL_NAME = 'approach3c_dino_multi_path_scoring'
C_RUN = C_RUN = cfg.CHECKPOINT_DIR / 'approach3cv1_unfreeze2_4_6_dino_pairwise_feature_fusion_20261127_125555_454041'  # Completed 3Cv1 checkpoint run folder.
WEIGHTS = (0.5, 0.25, 0.25)  # joint, crop, disease
cfg.NUM_WORKERS = 0
cfg.set_seed(cfg.SEED)

# 2. Data

Development uses the common training-derived candidate order.

In [2]:
pairs = get_pairs()

# 3. Model

Restore the original trained architectures and combine independently normalized candidate scores.

In [3]:
MODEL = 'vit_base_patch14_dinov2.lvd142m'

def c_components(HP):
    def snapshot(repository,revision,allow_download,patterns):
        from huggingface_hub import snapshot_download
        path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
        return path

    class Dino(nn.Module):
        """Exact two-independent-tower initialization from ToTnki's released runner."""
        def __init__(self,config,device,allow_download=False,revision=None):
            super().__init__()
            import timm
            path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                          ['config.json','model.safetensors','pytorch_model.bin'])
            weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
            def pinned_source_model(classes):
                # Same source factory, with only an explicit weight-file binding added.
                model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                        pretrained_cfg_overlay={'file':str(weights)})
                model.requires_grad_(False);model.head.requires_grad_(True)
                return model
            self.crop=pinned_source_model(cfg.NUM_CROPS)
            self.disease=pinned_source_model(cfg.NUM_DISEASES)
            self.augmentation_rng_state=torch.get_rng_state().clone()
            self.width=self.crop.num_features
            self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
                architecture=MODEL,width=self.width,independent_visual_towers=2,
                initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
                preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
            self.to(device)
        def policy(self, depth):
            self.requires_grad_(False)
            for tower in (self.crop, self.disease):
                tower.blocks[-depth:].requires_grad_(True)
                tower.set_grad_checkpointing(True)

        def features(self,pixels):
            return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                    self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
        def forward(self,pixels):return self.features(pixels)

    def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))


    RECIPE = dict(synthetic_pairs='observed_training_only', loss='moving_weighted_sum')

    class Model(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone
            width, hidden, dim = backbone.width, HP['hidden_dim'], HP['feature_dim']
            self.crop_projection = projection(width, hidden, dim)
            self.disease_projection = projection(width, hidden, dim)
            self.crop_aux = nn.Linear(dim, cfg.NUM_CROPS)
            self.disease_aux = nn.Linear(dim, cfg.NUM_DISEASES)
            self.attention = nn.MultiheadAttention(dim, HP['fusion_attention_heads'], batch_first=True)
            self.image_fusion = nn.Sequential(nn.LayerNorm(2 * dim), nn.Linear(2 * dim, dim), nn.GELU())
            self.crop_embedding = nn.Embedding(cfg.NUM_CROPS, dim)
            self.disease_embedding = nn.Embedding(cfg.NUM_DISEASES, dim)
            self.pair_fusion = nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU())

        def features(self, images):
            crop, disease = self.backbone(images)
            return self.crop_projection(crop), self.disease_projection(disease)

        def score(self, crop, disease, candidate_pairs):
            combined = crop + disease
            tokens = torch.stack((crop + combined, disease + combined), dim=1)
            attended, _ = self.attention(tokens, tokens, tokens, need_weights=False)
            residuals = torch.stack((disease, crop), dim=1)
            image = self.image_fusion((attended + residuals).flatten(1))
            crop_ids, disease_ids = candidate_pairs.unbind(1)
            candidates = self.pair_fusion(torch.cat(
                (self.crop_embedding(crop_ids), self.disease_embedding(disease_ids)), dim=1))
            return image @ candidates.T


    def make_model(depth, metadata=None):
        cfg.set_seed(cfg.SEED)
        metadata = metadata or {}
        backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
        backbone.policy(depth)
        cfg.set_seed(cfg.SEED)
        model = Model(backbone).to(cfg.DEVICE)
        torch.set_rng_state(backbone.augmentation_rng_state)
        return model, dict(dino=backbone.metadata)


    def make_predict(model, candidate_pairs):
        candidates = candidate_pairs.to(cfg.DEVICE)

        @torch.no_grad()
        def predict(images):
            model.eval()
            return model.score(*model.features(images.to(cfg.DEVICE)), candidates)
        return predict
    return make_model, RECIPE

def load_run(folder, model_name, family, recipe):
    run = Path(folder)
    selection = json.loads((run / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(run / selection['checkpoint'], map_location='cpu', weights_only=True)
    expected = dict(model_name=model_name, family=family, debug=False, recipe=recipe)
    if any(record.get(key) != value for record in (selection, checkpoint)
           for key, value in expected.items()):
        raise ValueError(f'Expected a full run from {model_name}.')
    if (not torch.equal(checkpoint['pairs'], pairs) or selection['development_pairs'] != pairs.tolist()
            or any(selection[key] != checkpoint[key] for key in ('hp', 'depth', 'best_epoch', 'best_val'))):
        raise ValueError('Checkpoint and selection do not match.')
    hp = checkpoint['hp']
    if hp['backbone'] != MODEL or hp['norm'] != 'imagenet' or hp['seed'] != cfg.SEED:
        raise ValueError('Unexpected backbone, preprocessing or seed.')
    if checkpoint['depth'] not in hp['unfreeze_blocks'] or checkpoint['depth'] not in (2, 4, 6, 8):
        raise ValueError('Checkpoint depth does not match the sweep.')
    train, val, _, _ = get_loaders(hp['batch_size'], norm=hp['norm'], debug=False)
    if (selection['train_images'] != train.dataset.frame.image.tolist()
            or selection['validation_images'] != val.dataset.frame.image.tolist()):
        raise ValueError('Shared train/validation membership changed.')
    return checkpoint


def freeze(model, checkpoint, metadata):
    if metadata != checkpoint['metadata']:
        raise ValueError('Pretrained model identity changed.')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval().requires_grad_(False)
    for tower in (model.backbone.crop, model.backbone.disease):
        tower.set_grad_checkpointing(False)
    return model


def soft_vote(scores, weights):
    return sum(weight * branch.float().softmax(dim=1)
               for weight, branch in zip(weights, scores, strict=True))

# 4. Training

No retraining. The weights are fixed before official data access and have not been optimized on any test results.

In [4]:
if len(WEIGHTS) != 3 or min(WEIGHTS) < 0 or not math.isclose(sum(WEIGHTS), 1.0):
    raise ValueError('Provide nonnegative weights summing to one.')
frozen_weights = tuple(WEIGHTS)

# 5. Evaluation

Enable once the input runs and weights are fixed. Every image uses the same scorer across all 333 candidates.

In [7]:
RUN_FINAL_EVALUATION = True
result = None
if RUN_FINAL_EVALUATION:
    if C_RUN is None:
        raise ValueError('Set C_RUN to the completed 3Cv1 checkpoint folder.')
    c_ckpt = load_run(C_RUN, 'approach3cv1_unfreeze2_4_6_dino_pairwise_feature_fusion', 'C',
                      dict(synthetic_pairs='observed_training_only', loss='moving_weighted_sum'))
    c_factory, _ = c_components(c_ckpt['hp'])
    c_model, c_meta = c_factory(c_ckpt['depth'], c_ckpt['metadata'])
    freeze(c_model, c_ckpt, c_meta)
    c_depth = c_ckpt['depth']
    batch_size = c_ckpt['hp']['batch_size']
    del c_ckpt

    final_pairs = get_pairs(final_evaluation=True)
    candidates = final_pairs.to(cfg.DEVICE)
    crop_ids, disease_ids = candidates.unbind(1)

    @torch.no_grad()
    def predict(images):
        crop, disease = c_model.features(images.to(cfg.DEVICE))
        joint = c_model.score(crop, disease, candidates)
        crop_scores = c_model.crop_aux(crop)[:, crop_ids]
        disease_scores = c_model.disease_aux(disease)[:, disease_ids]
        return soft_vote((joint, crop_scores, disease_scores), frozen_weights)

    hp = dict(weights=list(frozen_weights), c_depth=c_depth)

    seen, unseen = get_final_test_loaders(batch_size, norm='imagenet', final_evaluation=True)
    run = MODEL_NAME + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    previous = cfg.RESULTS_DIR
    RESULTS = previous / run
    RESULTS.mkdir(parents=True, exist_ok=False)
    try:
        cfg.RESULTS_DIR = RESULTS
        result = evaluate(predict, seen, unseen, MODEL_NAME, hp, robust=True)
    finally:
        cfg.RESULTS_DIR = previous

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

{
  "model": "approach3c_dino_multi_path_scoring",
  "seen_top1": 72.85,
  "seen_top5": 90.8,
  "unseen_top1": 17.2,
  "unseen_top5": 52.56,
  "harmonic_mean": 27.83,
  "crop_acc": 93.9,
  "disease_acc": 74.35,
  "robustness": {
    "brightness": {
      "top1": 72.7,
      "drop": 0.15
    },
    "blur": {
      "top1": 71.0,
      "drop": 1.85
    },
    "noise": {
      "top1": 66.4,
      "drop": 6.45
    }
  }
}


# 6. Save

The shared evaluator writes metrics and confusion files.

In [8]:
if result is not None:
    print('Results:', RESULTS)

Results: C:\Users\TABLE #012\Desktop\COS30082-Plant-Disease-OOD\results\approach3c_dino_multi_path_scoring_20261127_151516_979841
